In [3]:
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import Pie,Page
from pyecharts.charts import Bar
#1.读取
df = pd.read_csv('data.csv',encoding='ISO-8859-1')
#2.清洗id并显式复制一份，避免警告
df1 = df.dropna(subset=['CustomerID']).copy()
df1.isna().sum()
#3.转换类型
df1['Quantity']=df1['Quantity'].astype(int)
#4.多条件筛选(记得加括号)
df1 = df1[(df1['UnitPrice'] > 0) & (df1['Quantity'] > 0)]
#5.计算总金额
df1['aggregate amount'] = df1['Quantity'] * df1['UnitPrice']
#6.将发票日期转换为datetime
df1['InvoiceDate'] = pd.to_datetime(df1['InvoiceDate'])
#7.RFM指标计算
df1['InvoiceDate'].max()
snapshot_data = (df1['InvoiceDate'].max() + pd.Timedelta(days=1))#设置标准日期
#按Customer分组，一次性算出R,F,M
rfm = df1.groupby('CustomerID').agg({'InvoiceDate':lambda x:(snapshot_data - x.max()).days,#计算天数差（R）
                                     'InvoiceNo':'count',  #订单次数（F）
                                     'aggregate amount':'sum',#总金额（M）
                                     })
rfm.rename(columns={
    'InvoiceDate':'Recency',
    'InvoiceNo':'Frequency',
    'aggregate amount':'Monetary'
}, inplace=True)
#数据标准化与特征分段
#分箱，分成5个级别
rfm['M_Score'] = pd.qcut(rfm['Monetary'],5,labels=[1,2,3,4,5])
rfm['F_Score'] = pd.qcut(rfm['Frequency'],5,labels=[1,2,3,4,5])
rfm['R_Score'] = pd.qcut(rfm['Recency'],5,labels=[5,4,3,2,1])
rfm['M_Score'] = rfm['M_Score'].astype(str)
rfm['F_Score'] = rfm['F_Score'].astype(str)
rfm['R_Score'] = rfm['R_Score'].astype(str)
#拼接,通过数字来分辨客户类型
rfm['RFM_Group'] = rfm['M_Score'] + rfm['F_Score'] + rfm['R_Score']
rfm['RFM_Group'].value_counts().head()
def professional_segment(row):
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    if r >= '4' and f >= '4' and m >= '4':
        return '重要价值客户(Champions)'
    elif r >= '4' and f < '4' and m >= '4':
        return '重要发展客户(Potential Loyalists)'
    elif r < '4' and f >= '4' and m >= '4':
        return '重要保持客户(Loyal Customers)'
    elif r < '4' and f < '4' and m >= '4':
        return '重要挽留客户(AtRisk)'
    elif r >= '4' and f < '4' and m < '4':
        return '新客户(New Customers)'
    elif r < '4' and f < '4' and m < '4':
        return '流失客户(Lost)'
    else:
        return '一般维持客户'
rfm['Segment'] = rfm.apply(professional_segment, axis=1)
print(rfm['Segment'].value_counts())
#找出消费最多的金主
segment_revenue = rfm.groupby('Segment')['Monetary'].sum()
#求出不同人群的消费金额占比
group_share = (segment_revenue / segment_revenue.sum()) * 100
print(f"各组占比总和为{group_share.sum()}%")
#数据回填到rfm表
rfm['Segment_Revenue_Share'] = rfm['Segment'].map(group_share)
rfm[['Segment','Monetary','Segment_Revenue_Share']].head()


#数据可视化
#创建饼图
data_pair = [list(i) for i in zip(group_share.index.tolist(),group_share.values.tolist())]
c = Pie()
c.add("",data_pair,
radius=["40%","75%"])
c.set_global_opts(title_opts=opts.TitleOpts(title="英国礼品店-RFM营收贡献占比"),legend_opts=opts.LegendOpts(orient="vertical",pos_top="middle",pos_right="2%",)
                  )
c.set_series_opts(label_opts=opts.LabelOpts(position="outside",formatter="{b}:{c}%"))
c.render("rfm_chart.html")

#进一步分析
rfm['M_Score'] = rfm['M_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm_mean = rfm.groupby('Segment').agg({'R_Score':'mean','F_Score':'mean','M_Score':'mean',})
rfm_mean2 = rfm.groupby('Segment').agg({'Recency':'mean','Frequency':'mean','Monetary':'mean',})

bar1 = Bar()
bar1.add_xaxis(rfm_mean2.index.tolist())
bar1.add_yaxis("平均消费金额",rfm_mean2['Monetary'].round(2).tolist())
bar1.reversal_axis()
bar1.set_global_opts(title_opts=opts.TitleOpts(title="各用户群体平均消费金额对比"))
bar1.render("rfm_bar_chart.html")

group_counts = rfm['Segment'].value_counts()
data_pair = [list(i) for i in zip(group_counts.index.tolist(),group_counts.values.tolist())]
c_counts = Pie()
c_counts.add("",data_pair,
radius=["40%","75%"])
c_counts.set_global_opts(title_opts=opts.TitleOpts(title="英国礼品店-RFM用户人数结构"),legend_opts=opts.LegendOpts(orient="vertical",pos_top="middle",pos_right="2%",))
c_counts.set_series_opts(label_opts=opts.LabelOpts(position="outside",formatter="{b}:{d}%({c%}"))
c_counts.render("rfm_chart.html")


page = Page(layout=Page.SimplePageLayout)
page.add(c_counts)
page.add(c)
page.add(bar1)
page.render("index.html")

rfm.to_csv('rfm_final_result.csv', index=False, encoding='utf-8-sig')
print('洗好的数据已经成功导出为:rfm_final_result.csv')

Segment
流失客户(Lost)                     1717
重要价值客户(Champions)               932
新客户(New Customers)              526
重要保持客户(Loyal Customers)         434
一般维持客户                          360
重要挽留客户(AtRisk)                  224
重要发展客户(Potential Loyalists)     145
Name: count, dtype: int64
各组占比总和为100.00000000000003%
洗好的数据已经成功导出为:rfm_final_result.csv
